# TON-IoT Preprocessing

This notebook implements the preprocessing pipeline

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from s20neHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

Import required libraries used throughout preprocessing

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 1. LOAD
from pathlib import Path
NOTEBOOK_DIR = Path.cwd()
print (NOTEBOOK_DIR)
DATASET_DIR = NOTEBOOK_DIR / 'drive' / 'MyDrive' / 'MLmodeling' / 'XAI' / 'Datasets' / 'TON_IoT Network'
BASE_DIR = NOTEBOOK_DIR / 'drive' / 'MyDrive' / 'MLmodeling' / 'XAI'
SPLITS_DIR = BASE_DIR / 'splits' / 'TON_IoT'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
csv_path = DATASET_DIR / 'train_test_network.csv'
if not csv_path.exists():
    raise FileNotFoundError(f"CSV not found: {csv_path}")
df = pd.read_csv(csv_path)
print(f"Loaded: {df.shape}")

/content
Loaded: (211043, 44)


In [ ]:
import plotly.express as px
if 'type' in df.columns:
  fig1 = px.histogram(df, x='type', color='type',
                    title="📊 Distribution of Attack Types",
                    template="plotly_dark", text_auto=True)

# Customize the layout for readability:
# - Rename x and y axes
# - Remove redundant legend since each color already corresponds to a unique type
  fig1.update_layout(xaxis_title="Attack Type",
                   yaxis_title="Count",
                   showlegend=False)

# Display the interactive chart
  fig1.show()

In [ ]:
if 'duration' in df.columns:
    fig1 = px.box(df, x='type', y='duration', color='type',
                  title="⏱️ Flow Duration by Attack Type",
                  template="plotly_dark", points='all')
    fig1.update_layout(xaxis_title="Attack Type", yaxis_title="Flow Duration")
    fig1.show()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# 🟨 Step 4.2 — Boxplot: Source Bytes vs Attack Type

# Check if 'src_bytes' column exists before plotting.
if 'src_bytes' in df.columns:
    fig2 = px.box(df, x='type', y='src_bytes', color='type',
                  title="📦 Source Bytes by Attack Type",
                  template="plotly_dark", points='all')
    fig2.update_layout(xaxis_title="Attack Type", yaxis_title="Source Bytes")
    fig2.show()

# 💡 Why we did this step:
# ------------------------
# ✅ Purpose:
#    To understand how much data is being sent (in bytes) from the source
#    during different types of attacks.

# ✅ Boxplots reveal:
#    - The **median** and **spread** of source byte traffic per attack type.
#    - **Outliers**, which may indicate unusually large data transfers.
#    - **Patterns** (e.g., DoS attacks sending high data volumes, while scans send very little).

# ✅ Insights we can get:
#    - Some attacks (like DDoS or brute force) generate heavy outgoing traffic.
#    - Others (like data exfiltration or XSS) might have more controlled byte patterns.

# ✅ Why it matters:
#    - Byte-based features often differentiate attack behaviors.
#    - Visualizing them helps detect unique “traffic signatures” for each attack.

# ✅ In summary:
#    This step gives a statistical and visual perspective on how much
#    outgoing data is associated with different attack categories.

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# Check if 'dst_bytes' column exists before creating the plot.
if 'dst_bytes' in df.columns:
    fig3 = px.box(df, x='type', y='dst_bytes', color='type',
                  title="📦 Destination Bytes by Attack Type",
                  template="plotly_dark", points='all')
    fig3.update_layout(xaxis_title="Attack Type", yaxis_title="Destination Bytes")
    fig3.show()

# 💡 Why we did this step:
# ------------------------
# ✅ Purpose:
#    To analyze how much data is being **received** (in bytes) by the destination
#    during different network attack types.

# ✅ What this boxplot reveals:
#    - The **distribution** (median, IQR) of incoming data per attack type.
#    - **Outliers** representing abnormally large data receptions (possible data floods or leaks).
#    - Differences in **data volume patterns** between attack categories.

# ✅ Insights you can extract:
#    - DoS and DDoS attacks may show **very high dst_bytes** due to massive response loads.
#    - Port scans or probes may show **very small dst_bytes**, as they only send/receive minimal data.
#    - Malware or infiltration traffic could have **moderate but consistent** byte sizes.

# ✅ Why it’s useful:
#    - Comparing source and destination bytes helps detect **traffic asymmetry**,
#      which is often a key indicator of malicious behavior.
#    - This can later guide **feature selection** for model training — for instance,
#      including `src_bytes`, `dst_bytes`, and their ratio.

# ✅ In summary:
#    This visualization highlights how much data each attack type tends to receive,
#    helping us differentiate attacks based on network data flow characteristics.


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# 2. DROP LEAKAGE COLUMNS
leakage_cols = ['type', 'src_ip', 'dst_ip']
df.drop(columns=[c for c in leakage_cols if c in df.columns], inplace=True, errors='ignore')
feature_log = {
    'dropped_leakage': leakage_cols,
    'reason': {
        'type': 'Directly encodes attack category — leakage',
        'src_ip': 'IP identifier, not a generalizable feature',
        'dst_ip': 'IP identifier, not a generalizable feature'
    }
}
print('Leakage columns dropped:', leakage_cols)

Leakage columns dropped: ['type', 'src_ip', 'dst_ip']


Drop columns that leak label information or are identifiers (IP, type, etc.)

In [ ]:
# 3. DROP HIGH-CARDINALITY COLUMNS
high_cardinality_cols = [
    'dns_query', 'ssl_subject', 'ssl_issuer', 'http_uri', 'http_user_agent',
    'http_orig_mime_types', 'http_resp_mime_types', 'weird_addl'
]
drop_cols = [c for c in high_cardinality_cols if c in df.columns]
df.drop(columns=drop_cols, inplace=True, errors='ignore')
feature_log['dropped_high_cardinality'] = drop_cols
feature_log['reason_high_cardinality'] = "Over 95% values are '-' placeholder — no signal"
print('High-cardinality columns dropped:', drop_cols)

High-cardinality columns dropped: ['dns_query', 'ssl_subject', 'ssl_issuer', 'http_uri', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_addl']


Remove high-cardinality text columns that are unlikely to generalize

In [ ]:
# 4. SEPARATE FEATURES AND LABEL
if 'label' not in df.columns:
    raise KeyError("Expected 'label' column in dataset")
X = df.drop(columns=['label'])
y = df['label']
print(f"\nFeatures: {X.shape[1]} | Label distribution:")
print(y.value_counts())
print(f"Attack ratio: {y.mean():.2%}")


Features: 32 | Label distribution:
label
1    161043
0     50000
Name: count, dtype: int64
Attack ratio: 76.31%


Separate features and label; perform basic label checks

In [ ]:
# 5. IDENTIFY COLUMN TYPES
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [
    'proto', 'service', 'conn_state', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected',
    'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'http_trans_depth',
    'http_method', 'http_version', 'weird_name', 'weird_notice'
]
cat_cols = [c for c in cat_cols if c in X.columns]
feature_log['numeric_features'] = num_cols
feature_log['categorical_features'] = cat_cols
feature_log['total_features_used'] = len(num_cols) + len(cat_cols)
print(f"\nNumeric features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")


Numeric features: 16
Categorical features: 16


Identify numeric and categorical columns and record feature lists

In [ ]:
# 6. TRAIN / VAL / TEST SPLIT
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.15,
    random_state=42,
    stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.1765,
    random_state=42,
    stratify=y_temp
)
print(f"\nSplit sizes:")
print(f"  Train: {len(X_train)} ({len(X_train)/len(X):.1%})")
print(f"  Val:   {len(X_val)}   ({len(X_val)/len(X):.1%})")
print(f"  Test:  {len(X_test)}  ({len(X_test)/len(X):.1%})")


Split sizes:
  Train: 147724 (70.0%)
  Val:   31662   (15.0%)
  Test:  31657  (15.0%)


In [ ]:
if 'label' in df.columns:

    # Create an interactive Pie chart using Plotly Express.
    # This visualizes the overall distribution of normal vs. attack traffic.
    #
    # Parameters explained:
    # ---------------------
    # - names='label': the categories shown on the pie chart are derived from the 'label' column.
    # - title: gives a descriptive title to the plot.
    # - template="plotly_dark": sets a dark mode theme for better contrast.
    # - hole=0.4: converts it into a donut-style chart (for aesthetics & clarity).
    fig2 = px.pie(df,
                  names='label',
                  title="🧩 Label Distribution (Benign vs Attack)",
                  template="plotly_dark",
                  hole=0.4)

    # Update the trace to display both the percentage and label inside each segment.
    fig2.update_traces(textposition='inside', textinfo='percent+label')

    # Display the chart interactively.
    fig2.show()

 Define preprocessing pipelines for numeric and categorical features and assemble the ColumnTransformer

In [ ]:
# 7. BUILD PREPROCESSING PIPELINE
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
], remainder='drop')

In [ ]:
# 8. FIT ON TRAIN ONLY — TRANSFORM ALL THREE
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)
print(f"\nProcessed shapes:")
print(f"  X_train_proc: {X_train_proc.shape}")
print(f"  X_val_proc:   {X_val_proc.shape}")
print(f"  X_test_proc:  {X_test_proc.shape}")


Processed shapes:
  X_train_proc: (147724, 98)
  X_val_proc:   (31662, 98)
  X_test_proc:  (31657, 98)


In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns

# 2️⃣ Compute the correlation matrix.
#    Each value represents how strongly two numeric features move together.
#    Range: -1 (perfect negative), 0 (no correlation), +1 (perfect positive).
corr = df[numeric_cols].corr()

# 3️⃣ Create an interactive heatmap using Plotly Express.
#    - 'Viridis' gives a nice gradient color scale.
#    - zmin/zmax fix the scale from -1 to +1 for consistency.
#    - The heatmap shows how features relate — useful for spotting redundancy or strong patterns.
fig = px.imshow(
    corr,
    color_continuous_scale='Viridis',
    title='🔗 Correlation Heatmap of Numeric Features',
    template='plotly_dark',
    zmin=-1, zmax=1
)

# 4️⃣ Adjust layout for better readability.
fig.update_layout(width=1000, height=800)

# 5️⃣ Show the heatmap.
fig.show()

Fit preprocessing pipeline on training data and transform validation/test

In [ ]:
# 9. SAVE EVERYTHING
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
save_path = SPLITS_DIR
X_train.to_csv(save_path / 'X_train.csv', index=False)
X_val.to_csv(save_path / 'X_val.csv',   index=False)
X_test.to_csv(save_path / 'X_test.csv',  index=False)
y_train.to_csv(save_path / 'y_train.csv', index=False)
y_val.to_csv(save_path / 'y_val.csv',   index=False)
y_test.to_csv(save_path / 'y_test.csv',  index=False)
import numpy as _np
_np.save(str(save_path / 'X_train_proc.npy'), X_train_proc)
_np.save(str(save_path / 'X_val_proc.npy'),   X_val_proc)
_np.save(str(save_path / 'X_test_proc.npy'),  X_test_proc)
joblib.dump(preprocessor, save_path / 'preprocessor.pkl')
import json
with open(save_path / 'feature_log.json', 'w') as f:
    json.dump(feature_log, f, indent=2)
print("\nAll splits and preprocessor saved.")
print("Preprocessing complete. Ready for model training.")


All splits and preprocessor saved.
Preprocessing complete. Ready for model training.
